In [ ]:
!pip install sunpy[all]
!pip install batman-package
!pip install numpy  scipy matplotlib astropy sunpy drms cupy-cuda12x

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. SETUP PARAMETERS (Validated MCMC Geometry)
# =========================================================
GRID_SIZE = 9600
R_SUN = 3200
center = GRID_SIZE / 2

MCMC_RP_RS = 0.02991
MCMC_IMPACT = 0.5796
R_VENUS = R_SUN * MCMC_RP_RS
y_venus = center + (R_SUN * MCMC_IMPACT)

# Claret (2000) Non-Linear Limb Darkening
a1, a2, a3, a4 = 0.616, -0.252, 0.825, -0.429

# =========================================================
# 2. BUILD THE STAR (Static Reference)
# =========================================================
y_grid, x_grid = np.ogrid[:GRID_SIZE, :GRID_SIZE]
r_grid = np.sqrt((x_grid - center)**2 + (y_grid - center)**2)
solar_disk = r_grid <= R_SUN

star_image = np.zeros((GRID_SIZE, GRID_SIZE))
mu = np.sqrt(1 - (r_grid[solar_disk] / R_SUN)**2)
star_image[solar_disk] = 1.0 - a1*(1.0 - mu**0.5) - a2*(1.0 - mu) - a3*(1.0 - mu**1.5) - a4*(1.0 - mu**2)

baseline_flux = np.sum(star_image)

# =========================================================
# 3. TRANSIT SIMULATION (Full Curve Sweep)
# =========================================================
# Using 61 points (ODD NUMBER) ensures one point hits the exact center
NUM_POINTS = 61
chord_len = R_SUN * np.sqrt(1 - MCMC_IMPACT**2)
x_positions = np.linspace(center - chord_len - 1.5*R_VENUS,
                          center + chord_len + 1.5*R_VENUS, NUM_POINTS)

flux_curve = []
SUB_PIXELS = 40  # High precision for accurate ppm extraction

print(f"Simulating {NUM_POINTS} points with sub-pixel integration...")

for i, x_v in enumerate(x_positions):
    # Dynamic bounding box for speed
    x_min, x_max = int(np.floor(x_v - R_VENUS - 2)), int(np.ceil(x_v + R_VENUS + 2))
    y_min, y_max = int(np.floor(y_venus - R_VENUS - 2)), int(np.ceil(y_venus + R_VENUS + 2))

    # Clip to grid
    x_min, x_max = max(0, x_min), min(GRID_SIZE, x_max)
    y_min, y_max = max(0, y_min), min(GRID_SIZE, y_max)

    box_w, box_h = x_max - x_min, y_max - y_min

    if box_w <= 0 or box_h <= 0:
        flux_curve.append(1.0)
        continue

    # Sub-pixel mask generation
    sub_y, sub_x = np.ogrid[y_min : y_max : box_h * SUB_PIXELS * 1j,
                            x_min : x_max : box_w * SUB_PIXELS * 1j]
    sub_dist = np.sqrt((sub_x - x_v)**2 + (sub_y - y_venus)**2)
    sub_mask = (sub_dist <= R_VENUS).astype(float)

    # Reshape and mean to anti-alias
    exact_coverage = sub_mask.reshape(box_h, SUB_PIXELS, box_w, SUB_PIXELS).mean(axis=(1, 3))

    # Calculate blocked flux in this specific frame
    star_patch = star_image[y_min:y_max, x_min:x_max]
    blocked_flux = np.sum(star_patch * exact_coverage)

    flux_curve.append((baseline_flux - blocked_flux) / baseline_flux)

# =========================================================
# 4. DATA EXTRACTION & PLOTTING
# =========================================================
max_dip_fraction = 1.0 - min(flux_curve)
max_dip_ppm = max_dip_fraction * 1e6
target_dip=993.615

plt.figure(figsize=(10, 6))
plt.plot(x_positions - center, flux_curve, color='firebrick', marker='.', markersize=4, label='Numerical Simulation')
plt.axhline(1.0, color='black', linestyle='--', alpha=0.5)
plt.title(f"Validated Transit Simulation (Max Depth: {max_dip_ppm:.1f} ppm)")
plt.xlabel("X Position (Pixels from Center)")
plt.ylabel("Normalized Flux")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("==========================================")
print("      FINAL VALIDATION SUMMARY")
print("==========================================")
print(f"Calculated Transit Depth: {max_dip_ppm:.4f} ppm")
print(f"MCMC Benchmark Target:    {target_dip}ppm")
print(f"Grid Discrepancy (ppm):   {abs(target_dip - max_dip_ppm):.4f} ppm")
print("==========================================")

## Above code tells us that pixelation ( after adding all pixels) the error is aorund 0.3396 ppm as compared to transit depth obtained from venus transit 2012 data. so will neglect this small error

In [ ]:
from scipy.optimize import minimize_scalar
import batman
import numpy as np

# =========================================================
# 1. THE TARGET DEPTH (From MCMC Benchmark)
# =========================================================
target_depth_fraction = 993.615e-6

# =========================================================
# 2. THE EXACT MCMC PARAMETERS
# =========================================================
params = batman.TransitParams()
params.t0 = 0.0
params.per = 224.7
params.a = 229.36134  # Updated a/Rs
b = 0.5796           # Updated Impact Parameter
params.inc = np.degrees(np.arccos(b / params.a))
params.ecc, params.w = 0., 90.
params.u, params.limb_dark = [0.616, -0.252, 0.825, -0.429], "nonlinear"

# Use 1001 points to ensure t=0.0 is exactly evaluated for max depth
t_fit = np.linspace(-0.2, 0.2, 1001)

# =========================================================
# 3. DEFINE THE OBJECTIVE FUNCTION
# =========================================================
def depth_error(rp_guess):
    params.rp = rp_guess
    sim_curve = batman.TransitModel(params, t_fit).light_curve(params)
    sim_depth = 1.0 - np.min(sim_curve)
    return (sim_depth - target_depth_fraction)**2

# =========================================================
# 4. EXECUTE SOLVER
# =========================================================
print(f"Targeting Empirical Depth: {target_depth_fraction*1e6:.3f} ppm")
result = minimize_scalar(depth_error, bounds=(0.01, 0.05), method='bounded')

print(f"Recovered Rp/Rs: {result.x:.5f}")

# =========================================================
# 5. CALCULATE THE RADIUS ERROR
# =========================================================
d_earth_sun = 1.0147   # Updated exact ephemeris
d_earth_venus = 0.2887 # Updated exact ephemeris
R_SUN_KM = 695700.0    # IAU 2015 standard
true_radius = 6131     # Accepted optical radius

calculated_radius = result.x * (d_earth_venus / d_earth_sun) * R_SUN_KM
discrepancy = calculated_radius - true_radius

print("==========================================")
print(f"Calculated Physical Radius:  {calculated_radius:.1f} km")
print(f"True Optical Radius:       {true_radius:.1f} km")
print(f"--> RADIUS DISCREPANCY:     {discrepancy:+.1f} km")
print("==========================================")

### the radius batman outputs is simply the effective opaque radius—the size of a solid black disk that blocks the exact same amount of light as your fuzzy, refractive planet. So our baseline radius at 993.615 ppm is 5919.9 km when there are no starspots

# Now we simulate Star spots
- using sub-pixel method in star spots to reduce discretization error
- following code gives corresponding transit depth for different percentages of starspot coverage on star surface, it generates scatter starspots each time so each time the transit depth is different , we will need to average it out

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.patches as patches

# =========================================================
# 1. SETUP PARAMETERS (Validated High-Res & MCMC)
# =========================================================
GRID_SIZE = 9600  # Resolution validated for 0.33 ppm error
R_SUN = 3200
center = GRID_SIZE / 2

MCMC_RP_RS = 0.02991  # Exact MCMC Median
MCMC_IMPACT = 0.5796  # Exact MCMC Median
R_VENUS = R_SUN * MCMC_RP_RS
y_venus = center + (R_SUN * MCMC_IMPACT)

# Limb Darkening (Claret 2000)
A = [0.616, -0.252, 0.825, -0.429]

# =========================================================
# 2. THERMODYNAMIC SPOT GENERATOR (Silva 2003)
# =========================================================
def get_spot_contrast(t_star=5778, t_spot=4000, wavelength_nm=617.3):
    """Calculates contrast ratio based on Planck's Law (HMI Filter @ 617.3nm)"""
    hc = 1240.0       # eV * nm
    k = 8.617e-5      # eV / K

    exp_star = hc / (wavelength_nm * k * t_star)
    exp_spot = hc / (wavelength_nm * k * t_spot)

    return (np.exp(exp_star) - 1.0) / (np.exp(exp_spot) - 1.0)

def apply_precision_spot(image, x_s, y_s, r_s, f_contrast):
    x0, x1 = int(x_s - r_s - 2), int(x_s + r_s + 2)
    y0, y1 = int(y_s - r_s - 2), int(y_s + r_s + 2)
    x0, x1 = max(0, x0), min(GRID_SIZE, x1)
    y0, y1 = max(0, y0), min(GRID_SIZE, y1)

    sub = 3  # Local sub-sampling for spot edges
    yy, xx = np.ogrid[y0:y1:complex(0, (y1-y0)*sub), x0:x1:complex(0, (x1-x0)*sub)]
    mask_sub = (np.sqrt((xx - x_s)**2 + (yy - y_s)**2) <= r_s).astype(float)
    mask = mask_sub.reshape(y1-y0, sub, x1-x0, sub).mean(axis=(1, 3))
    image[y0:y1, x0:x1] *= (1.0 - mask + mask * f_contrast)
    return image

# =========================================================
# 3. BUILD SPOTTED STELLAR SURFACE
# =========================================================
y_grid, x_grid = np.ogrid[:GRID_SIZE, :GRID_SIZE]
r_grid = np.sqrt((x_grid - center)**2 + (y_grid - center)**2)
solar_disk = r_grid <= R_SUN

star_image = np.zeros((GRID_SIZE, GRID_SIZE))
mu = np.sqrt(np.maximum(0, 1 - (r_grid[solar_disk] / R_SUN)**2))
star_image[solar_disk] = 1.0 - A[0]*(1-mu**0.5) - A[1]*(1-mu) - A[2]*(1-mu**1.5) - A[3]*(1-mu**2)

# Spot parameters
f_contrast = get_spot_contrast()
target_cov = 0.26  # 26% unocculted coverage
current_cov = 0
total_sun_pix = np.sum(solar_disk)
margin = R_VENUS + 100 # Safe distance from planet path
clean_star_flux = np.sum(star_image)

print(f"Populating starspots (Contrast: {f_contrast:.3f})...")
while current_cov < target_cov:
    r_s = np.random.randint(40, 120) # Scaled for 9600 grid
    y_s = np.random.randint(0, GRID_SIZE)
    # Avoid planet path (Y-axis)
    if y_venus - margin < y_s < y_venus + margin: continue
    x_s = np.random.randint(0, GRID_SIZE)

    if np.sqrt((x_s - center)**2 + (y_s - center)**2) + r_s < R_SUN:
        star_image = apply_precision_spot(star_image, x_s, y_s, r_s, f_contrast)
        current_cov += (np.pi * r_s**2) / total_sun_pix

baseline_flux = np.sum(star_image)
# Calculate actual physical coverage accounting for overlaps
actual_pixel_coverage = (1.0 - (np.sum(star_image) / np.sum(clean_star_flux))) / (1.0 - f_contrast)

# =========================================================
# 4. TRANSIT SWEEP (The Stress Test)
# =========================================================
NUM_POINTS = 61
chord = R_SUN * np.sqrt(1 - MCMC_IMPACT**2)
x_pos = np.linspace(center - chord - 1.5*R_VENUS, center + chord + 1.5*R_VENUS, NUM_POINTS)
flux_curve = []
SUB = 40  # Matched to your 0.33 ppm validation run

print(f"Generating light curve (Sub-pixels: {SUB})...")
for xv in x_pos:
    x0, x1 = int(xv - R_VENUS - 2), int(xv + R_VENUS + 2)
    y0, y1 = int(y_venus - R_VENUS - 2), int(y_venus + R_VENUS + 2)
    x0, x1, y0, y1 = max(0, x0), min(GRID_SIZE, x1), max(0, y0), min(GRID_SIZE, y1)

    bw, bh = x1-x0, y1-y0
    if bw <= 0 or bh <= 0:
        flux_curve.append(1.0); continue

    sy, sx = np.ogrid[y0:y1:complex(0, bh*SUB), x0:x1:complex(0, bw*SUB)]
    mask = (np.sqrt((sx - xv)**2 + (sy - y_venus)**2) <= R_VENUS).astype(float)
    exact_cov = mask.reshape(bh, SUB, bw, SUB).mean(axis=(1, 3))
    blocked = np.sum(star_image[y0:y1, x0:x1] * exact_cov)
    flux_curve.append((baseline_flux - blocked) / baseline_flux)

# =========================================================
# 5. VIDEO RENDERING
# =========================================================
print("Rendering animation...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Downsample imshow for faster display on high-res grid
ax1.imshow(star_image[::4, ::4], cmap='afmhot', origin='lower')
planet_p = patches.Circle(((x_pos[0]/4), (y_venus/4)), (R_VENUS/4), color='black')
ax1.add_patch(planet_p)
ax1.axis('off')
ax1.set_title(f"Unocculted Coverage: 26%")

ax2.set_xlim(min(x_pos)-center, max(x_pos)-center)
ax2.set_ylim(min(flux_curve)-0.0001, 1.0001)
line, = ax2.plot([], [], 'r-', lw=2)
dot, = ax2.plot([], [], 'ko')
ax2.set_title("Spotted Transit Signal")
ax2.grid(True, alpha=0.3)

def update(f):
    planet_p.set_center(((x_pos[f]/4), (y_venus/4)))
    line.set_data(x_pos[:f+1]-center, flux_curve[:f+1])
    dot.set_data([x_pos[f]-center], [flux_curve[f]])
    return planet_p, line, dot

plt.close()
anim = FuncAnimation(fig, update, frames=NUM_POINTS, interval=80, blit=True)
display(HTML(anim.to_html5_video()))

print(f"Final Spotted Depth: {(1-min(flux_curve))*1e6:.3f} ppm")

## Above code generates vedio of transit for visualization and returns a transit dip for given area coverage percentage of star spots randomly scatered on star. this is just for one case so if you run above code for exactly same parametres, you will get slight deviations in final transit dip due to randomness. So in next part we will calculate 100 such radius for 100 cases in which star spots are randomly scatterd for given area and use median transit dip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import batman
import gc

# =========================================================
# 1. SETUP PARAMETERS (High-Res 9600 Grid)
# =========================================================
GRID_SIZE, R_SUN = 9600, 3200
center = GRID_SIZE / 2

MCMC_RP_RS, MCMC_IMPACT = 0.02991, 0.5796
R_VENUS = R_SUN * MCMC_RP_RS
y_venus = center + (R_SUN * MCMC_IMPACT)
A = [0.616, -0.252, 0.825, -0.429]

D_EARTH_SUN, D_EARTH_VENUS = 1.0147, 0.2887
R_SUN_KM = 695700.0

params = batman.TransitParams()
params.t0, params.per, params.a = 0.0, 224.7, 229.36134
params.inc = np.degrees(np.arccos(MCMC_IMPACT / params.a))
params.ecc, params.w, params.u, params.limb_dark = 0., 90., A, "nonlinear"
t_fit = np.linspace(-0.2, 0.2, 1001)

# =========================================================
# 2. PHYSICS & GEOMETRY FUNCTIONS
# =========================================================
def get_clean_star():
    y, x = np.ogrid[:GRID_SIZE, :GRID_SIZE]
    r = np.sqrt((x - center)**2 + (y - center)**2)
    mask = r <= R_SUN
    star = np.zeros((GRID_SIZE, GRID_SIZE))
    mu = np.sqrt(np.maximum(0, 1 - (r[mask] / R_SUN)**2))
    star[mask] = 1.0 - A[0]*(1-mu**0.5) - A[1]*(1-mu) - A[2]*(1-mu**1.5) - A[3]*(1-mu**2)
    return star, mask

def get_spot_contrast():
    hc, k = 1240.0, 8.617e-5
    ex_st = hc / (617.3 * k * 5778)
    ex_sp = hc / (617.3 * k * 4000)
    return (np.exp(ex_st) - 1.0) / (np.exp(ex_sp) - 1.0)

def add_spots_fast(clean_image, target_cov_frac, f_cont, seed):
    np.random.seed(seed)
    curr_cov, total_pix, margin = 0, np.sum(disk_mask), R_VENUS + 50

    # FIX: Use boolean map. True | True = True (solves overlap instantly with no float math)
    spot_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=bool)

    while curr_cov < target_cov_frac:
        r_s = np.random.randint(40, 120)
        y_s, x_s = np.random.randint(0, GRID_SIZE), np.random.randint(0, GRID_SIZE)

        if y_venus - margin < y_s < y_venus + margin: continue

        # FIX: Replaced expensive np.sqrt check with squared check
        if (x_s-center)**2 + (y_s-center)**2 < (R_SUN - r_s)**2:
            y0, y1 = int(y_s-r_s), int(y_s+r_s+1)
            x0, x1 = int(x_s-r_s), int(x_s+r_s+1)
            yy, xx = np.ogrid[y0:y1, x0:x1]

            # FIX: Removed sub-pixel engine. Standard boolean circle logic is >100x faster.
            m = ((xx - x_s)**2 + (yy - y_s)**2) <= r_s**2

            spot_map[y0:y1, x0:x1] |= m
            curr_cov += (np.pi * r_s**2) / total_pix

    spotted_image = np.copy(clean_image)
    # Apply contrast to all 'True' spot pixels at once
    spotted_image[spot_map] *= f_cont

    return spotted_image

def solve_r(depth_ppm):
    target_frac = depth_ppm / 1e6
    def err(rp):
        params.rp = rp
        sim_depth = 1.0 - np.min(batman.TransitModel(params, t_fit).light_curve(params))
        return (sim_depth - target_frac)**2
    res = minimize_scalar(err, bounds=(0.02, 0.05), method='bounded')
    return res.x * (D_EARTH_VENUS/D_EARTH_SUN) * R_SUN_KM

# =========================================================
# 3. PRE-COMPUTE MASKS & BASELINES
# =========================================================
f_contrast = get_spot_contrast()
clean_star_ref, disk_mask = get_clean_star()
clean_flux = np.sum(clean_star_ref)

x0, x1 = int(center-R_VENUS-2), int(center+R_VENUS+2)
y0, y1 = int(y_venus-R_VENUS-2), int(y_venus+R_VENUS+2)
SUB = 40 # High-res subpixel strictly kept for the planet mask only
sy, sx = np.ogrid[y0:y1:complex(0,(y1-y0)*SUB), x0:x1:complex(0,(x1-x0)*SUB)]
exact_p = (np.sqrt((sx-center)**2 + (sy-y_venus)**2) <= R_VENUS).astype(float).reshape(y1-y0, SUB, x1-x0, SUB).mean(axis=(1,3))

clean_blocked = np.sum(clean_star_ref[y0:y1, x0:x1] * exact_p)
clean_depth_ppm = (clean_blocked / clean_flux) * 1e6
BASELINE_RADIUS_KM = solve_r(clean_depth_ppm)

# =========================================================
# 4. CONSTANTS FOR ERROR PROPAGATION
# =========================================================
SIGMA_MCMC_RP_RS = 0.00031
SIGMA_MCMC_KM = SIGMA_MCMC_RP_RS * (D_EARTH_VENUS / D_EARTH_SUN) * R_SUN_KM

print(f"Propagating MCMC Statistical Uncertainty: ±{SIGMA_MCMC_KM:.2f} km")


# =========================================================
# 5. STATISTICAL ANCHOR EXECUTION
# =========================================================
NUM_TRIALS = 500
anchors_pct = [0, 5, 10, 15, 20, 25, 30]

median_inflation = []
topo_std_list = []
total_std_list = []

print(f"Starting execution. Baseline Radius established at {BASELINE_RADIUS_KM:.1f} km.")

for anchor in anchors_pct:
    print(f"Processing Anchor {anchor}% Coverage...")

    if anchor == 0:
        median_depth = clean_depth_ppm
        r_apparent = BASELINE_RADIUS_KM
        sig_topo = 0.0
    else:
        depths = []
        target_frac = anchor / 100.0
        for i in range(NUM_TRIALS):
            spotted_star = add_spots_fast(np.copy(clean_star_ref), target_frac, f_contrast, seed=(anchor * 1000) + i)
            base_flux = np.sum(spotted_star)
            blocked = np.sum(spotted_star[y0:y1, x0:x1] * exact_p)
            depths.append((blocked / base_flux) * 1e6)

        median_depth = np.median(depths)
        std_depth = np.std(depths)
        r_apparent = solve_r(median_depth)
        sig_topo = std_depth * (r_apparent / (2 * median_depth))

    inflation = r_apparent - BASELINE_RADIUS_KM
    median_inflation.append(inflation)
    topo_std_list.append(sig_topo)

    sig_sys_mcmc = SIGMA_MCMC_KM * (r_apparent / BASELINE_RADIUS_KM)
    sig_total = np.sqrt(sig_topo**2 + sig_sys_mcmc**2)
    total_std_list.append(sig_total)

# =========================================================
# 6. UNIFIED PLOTTING (Total Error Only)
# =========================================================
plt.figure(figsize=(12, 7))


med_arr = np.array(median_inflation)

plt.fill_between(anchors_pct,
                 med_arr - np.array(total_std_list),
                 med_arr + np.array(total_std_list),
                 color='gray', alpha=0.3, label=r'Total Uncertainty ($\sigma_{total}$)')

plt.plot(anchors_pct, median_inflation, 'o-', color='navy', lw=2, label='Median Inflation')

plt.axhline(0, color='black', lw=1, ls='--')
plt.xlim(0, max(anchors_pct) + 2)
plt.ylim(-80, max(med_arr + np.array(total_std_list)) * 1.05)

plt.title("Venus Apparent Radius Inflation vs. Stellar Activity", fontsize=14)
plt.xlabel("Actual Starspot Area Coverage (%)", fontsize=12)
plt.ylabel(r"$\Delta R_{Venus}$ (km)", fontsize=12)
plt.grid(True, which='both', linestyle=':', alpha=0.4)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Error propagation for Venus

In [ ]:
import numpy as np

# =========================================================
# RIGOROUS 26% EVALUATION (Run after master Venus loop)
# =========================================================
target_coverage = 0.26
NUM_TRIALS = 500

print(f"\n--- Running Rigorous {target_coverage*100}% Anchor ({NUM_TRIALS} Trials) ---")

depths_26 = []

# 1. Run the ensemble
for i in range(NUM_TRIALS):
    spotted_star = add_spots_fast(np.copy(clean_star_ref), target_coverage, f_contrast, seed=(26000 + i))
    base_flux = np.sum(spotted_star)
    blocked = np.sum(spotted_star[y0:y1, x0:x1] * exact_p)
    depths_26.append((blocked / base_flux) * 1e6)
    if (i+1) % 50 == 0: print(f"Completed {i+1}/{NUM_TRIALS}...")

# 2. Extract asymmetric percentiles
ppm_16 = np.percentile(depths_26, 16)
ppm_50 = np.percentile(depths_26, 50)
ppm_84 = np.percentile(depths_26, 84)

# 3. Convert to Apparent Radii (Inversion )
r_16 = solve_r(ppm_16)
r_50 = solve_r(ppm_50)
r_84 = solve_r(ppm_84)

# 4. Calculate Inflation
inflation_med = r_50 - BASELINE_RADIUS_KM

# 5. Calculate Topological Error (Asymmetric bounds from percentiles)
topo_err_lower = r_50 - r_16
topo_err_upper = r_84 - r_50

# 6. Calculate Systemic MCMC Error (Dynamically Scaled)
sig_sys_scaled = SIGMA_MCMC_KM * (r_50 / BASELINE_RADIUS_KM)

# 7. Total Error in Quadrature
total_err_lower = np.sqrt(topo_err_lower**2 + sig_sys_scaled**2)
total_err_upper = np.sqrt(topo_err_upper**2 + sig_sys_scaled**2)

print("\n==========================================")
print(f"Median Transit Depth:     {ppm_50:.2f} ppm")
print(f"Radius Inflation:         +{inflation_med:.1f} km")
print(f"Topological Noise (1σ):  +{topo_err_upper:.1f} / -{topo_err_lower:.1f} km")
print(f"Scaled MCMC Baseline:     ±{sig_sys_scaled:.1f} km")
print("---------------------------------------------------------")
print(f"FINAL RESULT FOR REPORT:  +{inflation_med:.1f} (+{total_err_upper:.1f} / -{total_err_lower:.1f}) km")
print("==========================================")

# Applying simulation to trappist-1 e




In [ ]:
''' Extracting 4 -parametre limb darkening parametres from ExoTiC-LD (database 3.1.2;
D. Grant & H. R. Wakeford 2024), by using trilinear
interpolation between the most TRAPPIST-1-like PHOENIX
models. as given in Allen et al (2026)'''
!pip install exotic-ld
from exotic_ld import StellarLimbDarkening

# 1. Define TRAPPIST-1 Stellar Parameters
sld = StellarLimbDarkening(
    M_H=0.04,          # Metallicity of TRAPPIST-1
    Teff=2566,         # Effective Temperature in Kelvin
    logg=5.2,          # Surface Gravity
    ld_model='phoenix', # <--- THE FIX: Explicitly call the PHOENIX grid
    ld_data_path='exotic_ld_data'
)

# 2. Compute the 4-parameter non-linear coefficients for JWST
print("Downloading PHOENIX stellar grids and computing coefficients...")
c1, c2, c3, c4 = sld.compute_4_parameter_non_linear_ld_coeffs(
    wavelength_range=[5500, 53800], # Wavelengths in Angstroms (0.55 to 5.38 um)
    mode='JWST_NIRSpec_Prism'
)

print("\n--- CORRECTED TRAPPIST-1 JWST Limb Darkening Coefficients ---")
print(f"a1 = {c1:.4f}")
print(f"a2 = {c2:.4f}")
print(f"a3 = {c3:.4f}")
print(f"a4 = {c4:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import batman
import gc

# =========================================================
# 1. SETUP PARAMETERS (TRAPPIST-1e Geometry)
# Citation: Agol et al. 2021 (The Planetary Science Journal)
# =========================================================
GRID_SIZE = 9600
R_STAR = 3200
center = GRID_SIZE / 2

# Physical parameters for TRAPPIST-1e (Agol et al. 2021)
R_STAR_KM = 0.1192 * 695700.0  # Table 7
R_PLANET_KM = 0.920 * 6371.0   # Table 6
RP_RS = 0.07079                # Table 5
IMPACT_PARAM = 0.191           # Table 5
A_RS = 52.855                  # Table 5
PERIOD = 6.101013              # Table 2

R_PLANET = R_STAR * RP_RS
y_planet = center + (R_STAR * IMPACT_PARAM)

# TRAPPIST-1 Limb Darkening (Non-Linear Approximation)

a1 = 1.6837
a2 = -1.5647
a3 = 1.0443
a4 = -0.2981
# =========================================================
# 1b. CALCULATE TRUE TARGET DEPTH WITH BATMAN
# =========================================================
params = batman.TransitParams()
params.t0, params.per, params.a = 0.0, PERIOD, A_RS
params.rp = RP_RS
params.inc = np.degrees(np.arccos(IMPACT_PARAM / params.a))
params.ecc, params.w = 0., 90.
params.u, params.limb_dark = [a1, a2, a3, a4], "nonlinear"

# Get exact mid-transit depth analytically
analytical_curve = batman.TransitModel(params, np.array([0.0])).light_curve(params)
TARGET_DEPTH_PPM = (1.0 - analytical_curve[0]) * 1e6

# =========================================================
# 2. BUILD THE STAR (Static Reference)
# =========================================================
y_grid, x_grid = np.ogrid[:GRID_SIZE, :GRID_SIZE]
r_grid = np.sqrt((x_grid - center)**2 + (y_grid - center)**2)
stellar_disk = r_grid <= R_STAR

star_image = np.zeros((GRID_SIZE, GRID_SIZE))
mu = np.sqrt(np.maximum(0, 1 - (r_grid[stellar_disk] / R_STAR)**2))
star_image[stellar_disk] = 1.0 - a1*(1.0 - mu**0.5) - a2*(1.0 - mu) - a3*(1.0 - mu**1.5) - a4*(1.0 - mu**2)

baseline_flux = np.sum(star_image)

# =========================================================
# 3. TRANSIT SIMULATION (Full Curve Sweep)
# =========================================================
NUM_POINTS = 201
chord_len = R_STAR * np.sqrt(1 - IMPACT_PARAM**2)
x_positions = np.linspace(center - chord_len - 1.5*R_PLANET,
                          center + chord_len + 1.5*R_PLANET, NUM_POINTS)

flux_curve = []
SUB_PIXELS = 30  # High precision for accurate ppm extraction

print(f"Simulating {NUM_POINTS} points for TRAPPIST-1e with {SUB_PIXELS}x sub-pixel integration...")

for i, x_v in enumerate(x_positions):
    # Dynamic bounding box for speed
    x_min, x_max = int(np.floor(x_v - R_PLANET - 2)), int(np.ceil(x_v + R_PLANET + 2))
    y_min, y_max = int(np.floor(y_planet - R_PLANET - 2)), int(np.ceil(y_planet + R_PLANET + 2))

    # Clip to grid
    x_min, x_max = max(0, x_min), min(GRID_SIZE, x_max)
    y_min, y_max = max(0, y_min), min(GRID_SIZE, y_max)

    box_w, box_h = x_max - x_min, y_max - y_min

    if box_w <= 0 or box_h <= 0:
        flux_curve.append(1.0)
        continue

    # Sub-pixel mask generation (Using complex() for robust integer bounds)
    sub_y, sub_x = np.ogrid[y_min : y_max : complex(0, box_h * SUB_PIXELS),
                            x_min : x_max : complex(0, box_w * SUB_PIXELS)]

    sub_dist = np.sqrt((sub_x - x_v)**2 + (sub_y - y_planet)**2)
    sub_mask = (sub_dist <= R_PLANET).astype(float)

    # Reshape and mean to anti-alias
    exact_coverage = sub_mask.reshape(box_h, SUB_PIXELS, box_w, SUB_PIXELS).mean(axis=(1, 3))

    # Calculate blocked flux in this specific frame
    star_patch = star_image[y_min:y_max, x_min:x_max]
    blocked_flux = np.sum(star_patch * exact_coverage)

    flux_curve.append((baseline_flux - blocked_flux) / baseline_flux)

    # Aggressively clear massive arrays from RAM
    del sub_y, sub_x, sub_dist, sub_mask, exact_coverage, star_patch
    gc.collect()

# =========================================================
# 4. DATA EXTRACTION & PLOTTING
# =========================================================
max_dip_fraction = 1.0 - min(flux_curve)
max_dip_ppm = max_dip_fraction * 1e6

plt.figure(figsize=(10, 6))
plt.plot(x_positions - center, flux_curve, color='navy', marker='.', markersize=4, label='Numerical Grid Simulation')
plt.axhline(1.0, color='black', linestyle='--', alpha=0.5)
plt.title(f"TRAPPIST-1e Validated Transit (Max Depth: {max_dip_ppm:.1f} ppm)")
plt.xlabel("X Position (Pixels from Center)")
plt.ylabel("Normalized Flux")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("==========================================")
print("       FINAL VALIDATION SUMMARY")
print("==========================================")
print(f"Calculated Grid Depth:    {max_dip_ppm:.4f} ppm")
print(f"BATMAN Benchmark Target:  {TARGET_DEPTH_PPM:.4f} ppm")
print(f"Grid Discrepancy (ppm):   {TARGET_DEPTH_PPM - max_dip_ppm:.4f} ppm")
print("==========================================")

In [ ]:
from scipy.optimize import minimize_scalar
import batman
import numpy as np

# =========================================================
# 1. SETUP OBSERVER MODEL
# =========================================================
R_SUN_KM = 695700.0

# Explicitly defining physical radii linked geometrically
R_STAR_KM = 0.1192 * R_SUN_KM
RP_RS = 0.07079
TRUE_RADIUS_KM = RP_RS * R_STAR_KM   # ~5870.4 km

params = batman.TransitParams()
params.t0 = 0.0
params.per = 6.101013
params.a = 52.855
b = 0.191
params.inc = np.degrees(np.arccos(b / params.a))
params.ecc, params.w = 0., 90.

'''a1 = 1.6837
a2 = -1.5647
a3 = 1.0443
a4 = -0.2981 '''

# THE FIX: Match the exact Non-Linear LD used in your grid generation
params.u, params.limb_dark = [1.6837,-1.5647 , 1.0443  ,-0.2981], "nonlinear"

# High-resolution time window
t_fit = np.linspace(-0.02, 0.02, 1000)

# =========================================================
# 2. THE SOLVER FUNCTION
# =========================================================
def solve_apparent_radius(grid_depth_ppm):
    target_depth_fraction = grid_depth_ppm / 1e6

    def depth_error(rp_rs_guess):
        params.rp = rp_rs_guess
        sim_curve = batman.TransitModel(params, t_fit).light_curve(params)
        sim_depth = 1.0 - np.min(sim_curve)
        return (sim_depth - target_depth_fraction)**2

    # Solve for the apparent Rp/Rs ratio
    res = minimize_scalar(depth_error, bounds=(0.05, 0.10), method='bounded')

    apparent_radius_km = res.x * R_STAR_KM
    return apparent_radius_km

# =========================================================
# 3. QUICK TEST
# =========================================================
clean_grid_depth = 5626.1329

calculated_radius = solve_apparent_radius(clean_grid_depth)
inflation_error = calculated_radius - TRUE_RADIUS_KM

print(f"Grid Input Depth:           {clean_grid_depth:.4f} ppm")
print(f"Calculated Apparent Radius: {calculated_radius:.2f} km")
print(f"True Physical Radius:       {TRUE_RADIUS_KM:.2f} km")
print(f"--> SYSTEMATIC BASELINE ERROR: {inflation_error:+.2f} km")

### The Forward Math: In the calibration script, we told batman: "Make me a planet with exactly a 5870.43 km radius. What is the transit depth?" It answered: 5626 ppm.The Reverse Math (The Solver): In the solver script you just ran, we told the exact same batman model: "I have a transit depth of 5626 ppm. What is the radius?" It answered: 5870.36 km. Now our baseline is 5626 ppm which corresponds to 5870.43 km radius. Also since only difference of radius matters, error doesn't make a problem. It is basically showing that we are very accurate with what we did, we can further reduce error by improving grid size and subpixel if compuatation allows us, so we take baseline as 5870.43 km

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
t=2566
hc=1240
k=8.61*(10**(-5))
np.seterr(over='ignore')
x=np.linspace(0.0001,4000,100000)
y=(8*np.pi*hc)/((x**5) * (np.exp(hc/(x*k*t))-1))
plt.plot(x,y,'r')
plt.ylabel("$\mu(\lambda)$", fontsize=15)  # Spectral radiance
plt.xlabel("$\lambda$ (nm)", fontsize=15)  # Wavelength
plt.title("Plank's law for Trappist")
plt.grid(True)
plt.legend(['2566 K'])

In [ ]:
from scipy.integrate import quad

def f(x,t):
    return (8*np.pi*hc)/(x**5*(np.exp(hc/(x*k*t))-1))
y=f(x,t)
max_y_index=np.argmax(y)
# Retrieve the corresponding x-value
max_x=x[max_y_index]
print(f"The wavelength at which Plank's function shows maxima is {max_x} nm")

### Since the star's overall emission peaks at 1130.4 nm, it makes perfect physical sense to set our observation wavelength to 1.13 μm to see the contrast ratio at the brightest part of the star's spectrum. so at same wavelength using temperature of star and spot in plank't law we are getting contrast ratio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.patches as patches
import gc

# =========================================================
# 1. SETUP PARAMETERS (TRAPPIST-1e Geometry - Agol 2021)
# =========================================================
GRID_SIZE = 2400  # Low resolution since this code is just for visualisation purpose
R_STAR = 800
center = GRID_SIZE / 2

# Physical parameters from Agol et al. 2021 Tables 5 & 6
R_SUN_KM = 695700.0
R_EARTH_KM = 6371.0

# Using median posterior values for Planet e
RP_RS = 0.07079       # Table 5
IMPACT_PARAM = 0.191  # Table 5 (b/R*)
R_STAR_KM = 0.1192 * R_SUN_KM # Table 7

R_PLANET = R_STAR * RP_RS
y_planet = center + (R_STAR * IMPACT_PARAM)

# Physical "True" Radius for metadata display
TRUE_RADIUS_KM = RP_RS * R_STAR_KM # ~5870.43 km

# Juniper AB Limb Darkening (Non-Linear - UNTOUCHED)
A = [1.6837,-1.5647 , 1.0443  ,-0.2981]

# =========================================================
# 2. THERMODYNAMIC SPOT GENERATOR
# =========================================================
def get_spot_contrast(t_star=2566, t_spot=2200, wavelength_nm=1130.4):
    """
    Calculates thermodynamic spot contrast using Planck's Law at peak wavelength.
    T_star from Agol 2021 Table 7.
    """
    hc = 1240.0       # eV * nm
    k = 8.617e-5      # eV / K

    exp_star = hc / (wavelength_nm * k * t_star)
    exp_spot = hc / (wavelength_nm * k * t_spot)

    contrast_ratio = (np.exp(exp_star) - 1.0) / (np.exp(exp_spot) - 1.0)
    return contrast_ratio

def apply_precision_spot(image, x_s, y_s, r_s, f_contrast):
    x0, x1 = int(x_s - r_s - 2), int(x_s + r_s + 2)
    y0, y1 = int(y_s - r_s - 2), int(y_s + r_s + 2)
    x0, x1 = max(0, x0), min(GRID_SIZE, x1)
    y0, y1 = max(0, y0), min(GRID_SIZE, y1)

    sub = 3
    yy, xx = np.ogrid[y0:y1:complex(0, (y1-y0)*sub), x0:x1:complex(0, (x1-x0)*sub)]
    mask_sub = (np.sqrt((xx - x_s)**2 + (yy - y_s)**2) <= r_s).astype(float)
    mask = mask_sub.reshape(y1-y0, sub, x1-x0, sub).mean(axis=(1, 3))
    image[y0:y1, x0:x1] *= (1.0 - mask + mask * f_contrast)
    return image

# =========================================================
# 3. BUILD SPOTTED STELLAR SURFACE
# =========================================================
y_grid, x_grid = np.ogrid[:GRID_SIZE, :GRID_SIZE]
r_grid = np.sqrt((x_grid - center)**2 + (y_grid - center)**2)
stellar_disk = r_grid <= R_STAR

star_image = np.zeros((GRID_SIZE, GRID_SIZE))
mu = np.sqrt(np.maximum(0, 1 - (r_grid[stellar_disk] / R_STAR)**2))
star_image[stellar_disk] = 1.0 - A[0]*(1-mu**0.5) - A[1]*(1-mu) - A[2]*(1-mu**1.5) - A[3]*(1-mu**2)

clean_star_flux = np.sum(star_image)

f_contrast = get_spot_contrast()
target_cov = 0.26  ## spot coverage % age
current_cov = 0
total_star_pix = np.sum(stellar_disk)
margin = R_PLANET + 30

print(f"Populating starspots at contrast {f_contrast:.3f}...")
while current_cov < target_cov:
    r_s = np.random.randint(15, 45)
    y_s = np.random.randint(0, GRID_SIZE)
    if y_planet - margin < y_s < y_planet + margin: continue
    x_s = np.random.randint(0, GRID_SIZE)

    if np.sqrt((x_s - center)**2 + (y_s - center)**2) + r_s < R_STAR:
        star_image = apply_precision_spot(star_image, x_s, y_s, r_s, f_contrast)
        current_cov += (np.pi * r_s**2) / total_star_pix

baseline_flux = np.sum(star_image)
actual_pixel_coverage = (1.0 - (baseline_flux / clean_star_flux)) / (1.0 - f_contrast)

# =========================================================
# 4. TRANSIT SWEEP
# =========================================================
NUM_POINTS = 151
chord = R_STAR * np.sqrt(1 - IMPACT_PARAM**2)
x_pos = np.linspace(center - chord - 1.5*R_PLANET, center + chord + 1.5*R_PLANET, NUM_POINTS)
flux_curve = []
SUB = 10

print("Generating light curve...")
for xv in x_pos:
    x0, x1 = int(xv - R_PLANET - 2), int(xv + R_PLANET + 2)
    y0, y1 = int(y_planet - R_PLANET - 2), int(y_planet + R_PLANET + 2)
    x0, x1, y0, y1 = max(0, x0), min(GRID_SIZE, x1), max(0, y0), min(GRID_SIZE, y1)

    bw, bh = x1-x0, y1-y0
    if bw <= 0 or bh <= 0:
        flux_curve.append(1.0); continue

    sy, sx = np.ogrid[y0:y1:complex(0, bh*SUB), x0:x1:complex(0, bw*SUB)]
    mask = (np.sqrt((sx - xv)**2 + (sy - y_planet)**2) <= R_PLANET).astype(float)
    exact_cov = mask.reshape(bh, SUB, bw, SUB).mean(axis=(1, 3))
    blocked = np.sum(star_image[y0:y1, x0:x1] * exact_cov)
    flux_curve.append((baseline_flux - blocked) / baseline_flux)

    del sy, sx, mask, exact_cov
    gc.collect()

# =========================================================
# 5. VIDEO RENDERING
# =========================================================
print("Rendering video...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(star_image, cmap='afmhot', origin='lower')
planet_p = patches.Circle((x_pos[0], y_planet), R_PLANET, color='black')
ax1.add_patch(planet_p)
ax1.set_title(f"TRAPPIST-1 Spotted Surface ({target_cov*100:.1f}%)")
ax1.axis('off')

ax2.set_xlim(min(x_pos)-center, max(x_pos)-center)
ax2.set_ylim(min(flux_curve) - 0.0005, 1.0005)
ax2.set_title("Spotted Transit Light Curve")
ax2.set_xlabel("X Position of Planet e (Pixels)")
ax2.set_ylabel("Normalized Flux")
ax2.grid(True, alpha=0.3)

line, = ax2.plot([], [], 'navy', lw=2)
dot, = ax2.plot([], [], 'ko')

def update(f):
    planet_p.set_center((x_pos[f], y_planet))
    line.set_data(x_pos[:f+1]-center, flux_curve[:f+1])
    dot.set_data([x_pos[f]-center], [flux_curve[f]])
    return planet_p, line, dot

plt.close()
anim = FuncAnimation(fig, update, frames=NUM_POINTS, interval=40, blit=True)
display(HTML(anim.to_html5_video()))

print(f"Calculated Spotted Depth: {(1-min(flux_curve))*1e6:.1f} ppm")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import batman
import gc

# =========================================================
# 1. SETUP & CALIBRATION CONSTANTS (Agol et al. 2021)
# =========================================================
GRID_SIZE = 9600
R_STAR = 3200
center = GRID_SIZE / 2

# Physical Constants (Table 5, 6, & 7)
R_SUN_KM = 695700.0
R_STAR_KM = 0.1192 * R_SUN_KM
AGOL_RP_RS = 0.07079
TRUE_RADIUS_BASELINE = AGOL_RP_RS * R_STAR_KM  # 5870.43 km
AGOL_A_RS = 52.855
AGOL_IMPACT = 0.191
AGOL_PERIOD = 6.101013

R_PLANET = R_STAR * AGOL_RP_RS
y_planet = center + (R_STAR * AGOL_IMPACT)

# Non-Linear Limb Darkening (Fixed for consistency)
A = [1.6837,-1.5647 , 1.0443  ,-0.2981]

# Batman Template for Solver
params = batman.TransitParams()
params.t0, params.per, params.a = 0.0, AGOL_PERIOD, AGOL_A_RS
params.inc = np.degrees(np.arccos(AGOL_IMPACT / params.a))
params.ecc, params.w = 0., 90.
params.u, params.limb_dark = A, "nonlinear"
t_fit = np.linspace(-0.01, 0.01, 50)

# =========================================================
# 2. STABILIZED FUNCTIONS
# =========================================================
def get_spot_contrast(t_star=2566, t_spot=2200, wavelength_nm=1130.4):
    """Calculates contrast at peak stellar flux (1130.4 nm) in stable eV units."""
    hc, k = 1240.0, 8.617e-5
    exp_star = hc / (wavelength_nm * k * t_star)
    exp_spot = hc / (wavelength_nm * k * t_spot)
    return (np.exp(exp_star) - 1.0) / (np.exp(exp_spot) - 1.0)

def get_clean_star():
    y, x = np.ogrid[:GRID_SIZE, :GRID_SIZE]
    r = np.sqrt((x - center)**2 + (y - center)**2)
    mask = r <= R_STAR
    star = np.zeros((GRID_SIZE, GRID_SIZE))
    mu = np.sqrt(np.maximum(0, 1 - (r[mask] / R_STAR)**2))
    star[mask] = 1.0 - A[0]*(1-mu**0.5) - A[1]*(1-mu) - A[2]*(1-mu**1.5) - A[3]*(1-mu**2)
    return star, mask

def add_spots_fast(image, disk_mask, target_cov, f_cont):
    curr_cov, total_pix, margin = 0, np.sum(disk_mask), R_PLANET + 40
    while curr_cov < target_cov:
        # Scale random spot sizes up for the 9600 grid
        r_s = np.random.randint(60, 240)
        y_s, x_s = np.random.randint(0, GRID_SIZE), np.random.randint(0, GRID_SIZE)

        # Avoid the transit chord
        if y_planet - margin < y_s < y_planet + margin: continue

        # Only place if fully within the stellar disk
        if (x_s-center)**2 + (y_s-center)**2 < (R_STAR - r_s)**2:
            y0, y1 = int(y_s-r_s), int(y_s+r_s+1)
            x0, x1 = int(x_s-r_s), int(x_s+r_s+1)
            yy, xx = np.ogrid[y0:y1, x0:x1]
            m = (xx-x_s)**2 + (yy-y_s)**2 <= r_s**2
            image[y0:y1, x0:x1][m] *= f_cont
            curr_cov += (np.pi * r_s**2) / total_pix
    return image

def solve_r(depth_ppm):
    t_frac = depth_ppm / 1e6
    def err(rp):
        params.rp = rp
        return ((1.0 - np.min(batman.TransitModel(params, t_fit).light_curve(params))) - t_frac)**2

    # Solve for Rp/Rs, then convert directly to km
    res = minimize_scalar(err, bounds=(0.05, 0.10), method='bounded')
    return res.x * R_STAR_KM

# =========================================================
# 3. BASELINE CALIBRATION & MID-TRANSIT MASK
# =========================================================
f_contrast = get_spot_contrast()
clean_star_ref, disk_mask = get_clean_star()
clean_flux = np.sum(clean_star_ref)

x0, x1 = int(center-R_PLANET-2), int(center+R_PLANET+2)
y0, y1 = int(y_planet-R_PLANET-2), int(y_planet+R_PLANET+2)

# Ultra-high precision sub-pixel mask for 9600 grid
SUB = 30
sy, sx = np.ogrid[y0:y1:complex(0,(y1-y0)*SUB), x0:x1:complex(0,(x1-x0)*SUB)]
p_mask = (np.sqrt((sx-center)**2 + (sy-y_planet)**2) <= R_PLANET).astype(float)
exact_p = p_mask.reshape(y1-y0, SUB, x1-x0, SUB).mean(axis=(1,3))

# Calculate absolute clean baseline
blocked_clean = np.sum(clean_star_ref[y0:y1, x0:x1] * exact_p)
CLEAN_DEPTH_PPM = (blocked_clean / clean_flux) * 1e6

del sy, sx, p_mask
gc.collect()

# =========================================================
# =========================================================
# 4. STATISTICAL ANCHOR EXECUTION (The Correct Way)
# =========================================================
NUM_TRIALS = 500  # 500 trials per anchor for a smooth std
anchors_pct = [0, 5, 10, 15, 20, 25, 30]

median_inflation = []
topo_std_list = []
total_std_list = []




# MCMC Transit Depth Error (Agol et al. 2021 Table 7)
AGOL_RP_RS_ERR = 0.00028
SIGMA_MCMC_KM = AGOL_RP_RS_ERR * R_STAR_KM

print(f"Starting execution. Baseline Radius established at {TRUE_RADIUS_BASELINE:.1f} km.")

for anchor in anchors_pct:
    print(f"Processing Anchor {anchor}% Coverage...")

    if anchor == 0:
        # Feed the clean simulated depth to the solver to get the simulation baseline
        median_depth = CLEAN_DEPTH_PPM
        r_apparent = solve_r(median_depth)
        sig_topo = 0.0
    else:
        depths = []
        target_frac = anchor / 100.0
        for i in range(NUM_TRIALS):
            np.random.seed((anchor * 1000) + i)
            spotted_star = add_spots_fast(np.copy(clean_star_ref), disk_mask, target_frac, f_contrast)
            base_flux = np.sum(spotted_star)
            blocked = np.sum(spotted_star[y0:y1, x0:x1] * exact_p)
            depths.append((blocked / base_flux) * 1e6)

        median_depth = np.median(depths)
        std_depth = np.std(depths)
        r_apparent = solve_r(median_depth)
        sig_topo = std_depth * (r_apparent / (2 * median_depth))

    # Calculate Inflation using the TRUE baseline
    inflation = r_apparent - TRUE_RADIUS_BASELINE
    median_inflation.append(inflation)

    # Store Topological Noise
    topo_std_list.append(sig_topo)

    # Calculate Systematic Agol floor (Scaled by apparent radius)
    sig_sys_mcmc = SIGMA_MCMC_KM * (r_apparent / TRUE_RADIUS_BASELINE)

    # Total Uncertainty: Quadrature sum
    sig_total = np.sqrt(sig_topo**2 + sig_sys_mcmc**2)
    total_std_list.append(sig_total)

# =========================================================
# 5. UNIFIED PLOTTING
# =========================================================
plt.figure(figsize=(12, 7))

med_arr = np.array(median_inflation)

# Total Error Band
plt.fill_between(anchors_pct,
                 med_arr - np.array(total_std_list),
                 med_arr + np.array(total_std_list),
                 color='gray', alpha=0.3, label=r'Total Uncertainty ($\sigma_{total}$)')

# Median Line
plt.plot(anchors_pct, median_inflation, 'o-', color='navy', lw=2, label='Median Inflation')

plt.axhline(0, color='black', lw=1, ls='--')
plt.xlim(0, max(anchors_pct) + 2)
plt.ylim(-40, max(med_arr + np.array(total_std_list)) * 1.05)

plt.title("TRAPPIST-1e: Apparent Radius Inflation vs. Stellar Activity", fontsize=14)
plt.xlabel("Actual Starspot Area Coverage (%)", fontsize=12)
plt.ylabel(r"$\Delta R_{Apparent}$ (km)", fontsize=12)
plt.grid(True, which='both', linestyle=':', alpha=0.4)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Error analysis for TRAPPIST-1e

In [ ]:
import numpy as np

# =========================================================
# RIGOROUS 26% EVALUATION (Run after master TRAPPIST-1e loop)
# =========================================================
target_coverage = 0.26
NUM_TRIALS = 500

print(f"\n--- Running Rigorous {target_coverage*100}% Anchor TRAPPIST-1e ({NUM_TRIALS} Trials) ---")

depths_26 = []

# 1. Run the ensemble
for i in range(NUM_TRIALS):
    # Set the seed externally to match your TRAPPIST-1e master script logic
    np.random.seed(26000 + i)

    # Generate spotted star using TRAPPIST-specific arguments
    spotted_star = add_spots_fast(np.copy(clean_star_ref), disk_mask, target_coverage, f_contrast)
    base_flux = np.sum(spotted_star)
    blocked = np.sum(spotted_star[y0:y1, x0:x1] * exact_p)
    depths_26.append((blocked / base_flux) * 1e6)

    if (i+1) % 50 == 0: print(f"Completed {i+1}/{NUM_TRIALS}...")

# 2. Extract asymmetric percentiles
ppm_16 = np.percentile(depths_26, 16)
ppm_50 = np.percentile(depths_26, 50)
ppm_84 = np.percentile(depths_26, 84)

# 3. Convert to Apparent Radii using TRAPPIST-1e inversion solver
r_16 = solve_r(ppm_16)
r_50 = solve_r(ppm_50)
r_84 = solve_r(ppm_84)

# 4. Calculate Inflation (Using TRAPPIST-1e true baseline)
inflation_med = r_50 - TRUE_RADIUS_BASELINE

# 5. Calculate Topological Error (Asymmetric bounds from percentiles)
topo_err_lower = r_50 - r_16
topo_err_upper = r_84 - r_50

# 6. Calculate Systemic MCMC Error (Dynamically Scaled)
sig_sys_scaled = SIGMA_MCMC_KM * (r_50 / TRUE_RADIUS_BASELINE)

# 7. Total Error in Quadrature
total_err_lower = np.sqrt(topo_err_lower**2 + sig_sys_scaled**2)
total_err_upper = np.sqrt(topo_err_upper**2 + sig_sys_scaled**2)

print("\n==========================================")
print(f"Median Transit Depth:     {ppm_50:.2f} ppm")
print(f"Radius Inflation:         +{inflation_med:.1f} km")
print(f"Topological Noise (1σ):  +{topo_err_upper:.1f} / -{topo_err_lower:.1f} km")
print(f"Scaled MCMC Baseline:     ±{sig_sys_scaled:.1f} km")
print("---------------------------------------------------------")
print(f"FINAL RESULT FOR EQ 5.3:  +{inflation_med:.1f} (+{total_err_upper:.1f} / -{total_err_lower:.1f}) km")
print("==========================================")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. DENSITY MISCLASSIFICATION CALCULATION (For 0-30% Plot)
# =========================================================
# TRAPPIST-1e Mass (Agol et al. 2021 Table 5)
M_EARTH_KG = 5.972e24
MASS_T1e_EARTH = 0.692
MASS_T1e_KG = MASS_T1e_EARTH * M_EARTH_KG

# Volume calculation function
def calc_density(radius_km):
    volume_m3 = (4/3) * np.pi * (radius_km * 1000)**3
    density_kg_m3 = MASS_T1e_KG / volume_m3
    return density_kg_m3 / 1000  # Return in g/cm^3

# Baseline Density
true_density = calc_density(TRUE_RADIUS_BASELINE)

# Apparent Density across all anchors (Uses master loop arrays)
apparent_radii = TRUE_RADIUS_BASELINE + med_arr
apparent_densities = np.array([calc_density(r) for r in apparent_radii])

# Density Error Propagation (Derivative method for symmetric plot bands)
density_errors = apparent_densities * 3 * (np.array(total_std_list) / apparent_radii)

# =========================================================
# 2. PLOTTING THE DENSITY SHIFT
# =========================================================
plt.figure(figsize=(10, 6))

plt.fill_between(anchors_pct,
                 apparent_densities - density_errors,
                 apparent_densities + density_errors,
                 color='orange', alpha=0.3, label=r'Density Uncertainty ($1\sigma$)')

plt.plot(anchors_pct, apparent_densities, 'o-', color='darkred', lw=2, label='Apparent Bulk Density')

plt.axhline(true_density, color='black', lw=1, ls='--', label=f'True Density ({true_density:.2f} g/cm$^3$)')

plt.title("TRAPPIST-1e: Density Misclassification due to Stellar Spots", fontsize=14)
plt.xlabel("Actual Starspot Area Coverage (%)", fontsize=12)
plt.ylabel(r"Derived Bulk Density (g/cm$^3$)", fontsize=12)
plt.grid(True, alpha=0.4)

# Add manual Y-limits so the top error band at 0% doesn't hit the plot ceiling
upper_bound = true_density + density_errors[0] + 0.05
lower_bound = np.min(apparent_densities - density_errors) - 0.05
plt.ylim(lower_bound, upper_bound)

plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

# =========================================================
# 3. EXACT 26% DENSITY EXTRACTION (Using Rigorous Variables)
# =========================================================
# This uses r_50, total_err_upper, total_err_lower from your Rigorous script
dens_med = calc_density(r_50)

# Invert the errors: The upper radius error forces the lower density bound
dens_lower_bound = calc_density(r_50 + total_err_upper)
dens_upper_bound = calc_density(r_50 - total_err_lower)

err_dens_down = dens_med - dens_lower_bound
err_dens_up = dens_upper_bound - dens_med

print("\n==========================================")
print(f"RIGOROUS TRAPPIST-1e DENSITY AT 26% COVERAGE")
print(f"True Baseline Density:    {true_density:.2f} g/cm^3")
print(f"Apparent Density:         {dens_med:.2f} (+{err_dens_up:.2f} / -{err_dens_down:.2f}) g/cm^3")
print("==========================================")